In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    current_timestamp,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)


# ============================================
# STORAGE PATHS
# ============================================

bronze_root = (
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
)

silver_root = (
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/"
)

In [0]:
# ============================================
# FIXED DEPOSIT: BRONZE → SILVER
# ============================================
# CONFIGURATION & SCHEMA SETUP
# ============================================
CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "silver"
TABLE_NAME = "fixed_deposit"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"

# Ensure Silver schema exists inside your Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

# ============================================
# READ BRONZE DATA
# ============================================
fd_bronze_df = spark.read.parquet(f"{bronze_root}fd/")

# ============================================
# TRANSFORM & CLEANSE (BRONZE → SILVER)
# ============================================
fd_silver_df = (
    fd_bronze_df
    .select(
        trim(col("fd_id")).alias("fd_id"),
        trim(col("customer_id")).alias("customer_id"),
        trim(col("deposit_amount")).cast(DecimalType(18, 2)).alias("deposit_amount"),
        trim(col("interest_rate")).cast(DecimalType(5, 2)).alias("interest_rate"),
        to_date(trim(col("maturity_date")), "yyyy-MM-dd").alias("maturity_date"),
        upper(trim(col("fd_status"))).alias("fd_status")
    )
    .filter(col("fd_id").isNotNull())
    .dropDuplicates(["fd_id"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

# ============================================
# WRITE FRESH DATA (OVERWRITE PATH & TABLE)
# ============================================
# 1. Overwrite raw Delta files in ADLS
(
    fd_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_root}fd/")
)

# 2. Overwrite / Register managed Unity Catalog table
(
    fd_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

# ============================================
# VERIFICATION METRICS
# ============================================
record_count = fd_silver_df.count()
print(f"FD Silver Count: {record_count} records saved to '{FULL_TABLE_NAME}'.")